# Steam : analyse du marché du jeu vidéo pour Ubisoft



## 1. Chargement et inspection des données


Nous allons tout d'abord importer le fichier demandé par l'enoncé depuis le serveur S3.


In [0]:
from pyspark.sql import functions as F

steam_text_df = spark.read.text("s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json")
print(steam_text_df.count())


1


### Forme brute du fichier

Nous allons d'abord examiner comment il est ecrit et comment il se structure.

In [0]:
print(steam_text_df.select(F.length("value")).take(1)[0][0])


63688273


In [0]:
print(steam_text_df.select(F.substring("value", 1, 1000)).take(1)[0][0])

[{"id": "10", "data": {"appid": 10, "name": "Counter-Strike", "short_description": "Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.", "developer": "Valve", "publisher": "Valve", "genre": "Action", "tags": {"Action": 5426, "FPS": 4831, "Multiplayer": 3392, "Shooter": 3353, "Classic": 2784, "Team-Based": 1864, "First-Person": 1707, "Competitive": 1607, "Tactical": 1344, "e-sports": 1192, "1990's": 1191, "PvP": 881, "Old School": 769, "Military": 632, "Strategy": 614, "Survival": 304, "Score Attack": 289, "1980s": 266, "Assassin": 227, "Nostalgia": 131}, "type": "game", "categories": ["Multi-player", "Valve Anti-Cheat enabled", "Online PvP", "Shared/Split Screen PvP", "PvP"], "owners": "10,000,000 .. 20,000,000", "positive": 

Le fichier est constitué de plus de 63 000 000 de caracteres.
Il est sur une seule ligne et elle contient toute l'information.

### Lecture en JSON

In [0]:
steam_raw_df = spark.read.json("s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json")


### Schéma

Nous allons afficher le schémades données. Il permet de connaitre le nom et le type de chaque champ du bloc data. 


In [0]:
steam_raw_df.printSchema()


root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

On observe que le schema est très long parce que le bloc tags contient un champ par nom de tag existant dans le catalogue. C'est ce bloc qui occupe la plus grande partie du schema.

In [0]:
display(steam_raw_df.limit(10))


data,id
"List(10, List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP), 13990, Valve, 0, Action, https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513, 999, English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean, Counter-Strike, 5199, 10,000,000 .. 20,000,000, List(true, true, true), 201215, 999, Valve, 2000/11/1, 0, Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role., List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192), game, )",10
"List(1000000, List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud), 0, IndigoBlue Game Studio, 0, Action, Adventure, Indie, https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048, 999, English, Korean, Simplified Chinese, ASCENXION, 5, 0 .. 20,000, List(false, false, true), 27, 999, PsychoFlux Entertainment, 2021/05/14, 0, ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, 

Nous pouvons observer les deux colonnes principale emerger., nous devons maintenant deplier Data pour examiner son contenu.

### Dépliage du bloc data

On sort les champs du bloc data pour en faire des colonnes : une ligne par jeu, une colonne par information.

In [0]:
display(steam_raw_df.select("data.*").limit(10))


appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website
10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192)",game,
1000000,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)",0,IndigoBlue Game Studio,0,"Action, Adventure, Indie",https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048,999,"English, Korean, Simplified Chinese",ASCENXION,5,"0 .. 20,000","List(false, false, true)",27,999,PsychoFlux Entertainment,2021/05/14,0,"ASCENXION is a 2D shoot 'em up game where you explore the field to progress. P

In [0]:
display(steam_raw_df.select("data.*").drop("tags").limit(10))


appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,type,website
10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,game,
1000000,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)",0,IndigoBlue Game Studio,0,"Action, Adventure, Indie",https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048,999,"English, Korean, Simplified Chinese",ASCENXION,5,"0 .. 20,000","List(false, false, true)",27,999,PsychoFlux Entertainment,2021/05/14,0,"ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.",game,
1000010,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)",99,NEXT Studios,70,"Adventure, Indie, RPG, Strategy",https://cdn.akamai.steamstatic.com/steam/apps/1000010/header.jpg?t=1655724189,1999,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil",Crown Trick,646,"200,000 .. 500,000","List(false, false, true)",4032,599,"Team17, NEXT Studios",2020/10/16,0,"Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!",game,
1000030,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)",76,Vertigo Gaming Inc.,0,"Action, Indie, Simulation, Strategy",https://cdn.akamai.steamstatic.com/steam/apps/1000030/header.jpg?t=1660866300,1999,English,"Cook, Serve, Delicious! 3?!",115,"100,000 .. 200,000","List(false, true, true)",1575,1999,Vertigo Gaming Inc.,2020/10/14,0,"Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!",game,http://www.cookservedelicious.com
1000040,List(Single-player),0,DoubleC Games,0,"Action, Casual, Indie, Simulation",https://cdn.akamai.steamstatic.com/steam/apps/1000040/header.jpg?t=1627033870,199,Simplified Chinese,细胞战争,1,"0 .. 20,000","List(false, false, true)",0,199,DoubleC Games,2019/03/30,0,这是一款打击感十足的细胞主题游戏！操作简单但活下去却不简单，“你”作为侵入人体的细菌病毒，通过与细胞之间的战斗来获得基因变异点数和进入下一关的资格，每种细菌病毒都有独特的能力和攻击效果，你是否可以破坏五大器官并占领人体呢！？,game,
1000080,"List(Multi-player, Single-player, Steam Achievements, Full controller support, Steam Trading Cards)",3,IndieLeague Studio,60,"Action, Adventure, Indie, RPG",https://cdn.akamai.steamstatic.com/steam/apps/1000080/header.jpg?t=1667062553,1999,"Simplified Chinese, English, Traditional Chinese, Japanese, Korean",Zengeon,462,"100,000 .. 200,000","List(false, true, true)",1018,799,2P Games,2019/06/24,0,Zengeon is an anime infused Action RPG and Roguelite with a selection of unique characters and varying play-styles. Slaughter yo

In [0]:
steam_raw_df.count()


55691

L'ensemble de données est composé de 55 691 lignes.


## 2. Mise à plat et nettoyage

### Verification si l'identifiant est en double.

Chaque jeu porte son numéro deux fois : dans la colonne id, et dans le champ appid du bloc data. Avant de ne garder que le contenu du bloc data, on compte les jeux pour lesquels ces deux numéros diffèrent.


In [0]:
steam_raw_df.filter(F.col("id") != F.col("data.appid").cast("string")).count()


0

Aucun jeu n'a deux numéros différents : la colonne id répète le champ appid. On peut donc ne garder que le contenu du bloc data sans rien perdre.


### Mise à plat du bloc data

Nous allons maintenant sortir les champs du bloc data pour en faire des colonnes : 
- une ligne par jeu
- une colonne par information. 

La colonne des tags n'est pas pertinente et est écartée. L'analyse ne porte pas sur les tags.


In [0]:
games_df = steam_raw_df.select("data.*").drop("tags")
games_df.printSchema()


root
 |-- appid: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- platforms: struct (nullable = true)
 |    |-- linux: boolean (nullable = true)
 |    |-- mac: boolean (nullable = true)
 |    |-- windows: boolean (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- type: string (nullable = true)
 |-- website: string

Les informations du jeu sont maintenant representées par des colonnes. Toutefois nous observons que les colonnes sont sous format texte (string) alors meme qu'il peut s'agir de date, prix ou autre.
Elles devront donc etre converties.

### Conversion des prix et des remises
Nous allons d'abord convertir les 3 colonnes suivantes:
- le prix
- le prix initial
- la remise   

Ces trois données sont écrites en texte. De plus les prix sont en centimes : Ils doivent etre convertit en nombres puis divisé par 100 pour obtenir des dollars. 

La remise est un pourcentage, convertie en nombre entier. Nous conserverons les colonnes d'origine, et placerons les valeurs converties dans de nouvelles colonnes.


In [0]:
games_df = games_df \
    .withColumn("price_usd", F.col("price").cast("int") / 100) \
    .withColumn("initial_price_usd", F.col("initialprice").cast("int") / 100) \
    .withColumn("discount_pct", F.col("discount").cast("int"))

display(games_df.select("name", "price", "price_usd", "initialprice", "initial_price_usd", "discount", "discount_pct").limit(10))


name,price,price_usd,initialprice,initial_price_usd,discount,discount_pct
Counter-Strike,999,9.99,999,9.99,0,0
ASCENXION,999,9.99,999,9.99,0,0
Crown Trick,599,5.99,1999,19.99,70,70
"Cook, Serve, Delicious! 3?!",1999,19.99,1999,19.99,0,0
细胞战争,199,1.99,199,1.99,0,0
Zengeon,799,7.99,1999,19.99,60,60
干支セトラ 陽ノ卷｜干支etc. 陽之卷,1299,12.99,1299,12.99,0,0
Jumping Master(跳跳大咖),0,0.0,0,0.0,0,0
Cube Defender,299,2.99,299,2.99,0,0
Tower of Origin2-Worm's Nest,1399,13.99,1399,13.99,0,0


### Conversion de la date de sortie

Tout comme les prix, les dates ne sont pas au format souhaité, elles sont en texte et doivent etre converties.

La date de sortie est écrite en texte, sous la forme année/mois/jour. Avant de la convertir, on vérifie que toutes les dates ont la même forme : on regroupe les jeux selon le nombre de caractères de leur date, avec un exemple par groupe.


In [0]:
date_shapes_df = games_df \
    .groupBy(F.length(F.col("release_date")).alias("date_length")) \
    .agg(F.count("*").alias("games"), F.min(F.col("release_date")).alias("example")) \
    .orderBy("date_length")

display(date_shapes_df)


date_length,games,example
0,99,
7,123,2009/06
9,15732,1998/11/8
10,39737,1997/06/30



Nous observons 4 possibilités d'ecriture de date:
- aucune date (99 jeux)
- date sans le jour (123 jeux)
- date sur 9 caracteres (15 732 jeux)
- dates sur 10 caracteres (39 737 jeux)




#### Traitement des dates incomplètes

Nous avions 3 possibilités pour traiter les dates incompletes:


- 1) supprimer les jeux :
cela conduirait à une supression de données pouvant servir aux autres analyses: prix, genre ou plateforme, la ou la date ne compte pas.
- 2) introduire une date arbitraire: Cela introduirait de l'information inventée.
- 3) conserver l'information disponible et laisser vide le reste : c'est l'option que nous allons retenir.

Les dates interviennent dans notre etude pour evaluer des tendances annuelles. Seules l'année est donc pertinente.

Nous allons toutefois convertir les dates completes en vraies dates de façon a preserver l'information. Concernant les 123 dates sans jour, nous n'avons veritablement besoin que de l'année donc cela n'est pas problematique. Enfin, concernant les 99 jeux sans date, ils restent dnas la table avec une date vide.



Nous allons creer deux colones supplementaires: 
- la date convertie (uniquement pour les dates completes)
- l'année de parution (pour tous les jeux ayant au moins une année)

In [0]:
games_df = games_df \
    .withColumn("release_date_clean", F.when(F.length(F.col("release_date")) >= 9, F.to_date(F.col("release_date"), "yyyy/M/d"))) \
    .withColumn("release_year", F.when(F.length(F.col("release_date")) >= 7, F.substring(F.col("release_date"), 1, 4).cast("int")))

display(games_df.select("name", "release_date", "release_date_clean", "release_year").limit(10))


name,release_date,release_date_clean,release_year
Counter-Strike,2000/11/1,2000-11-01,2000
ASCENXION,2021/05/14,2021-05-14,2021
Crown Trick,2020/10/16,2020-10-16,2020
"Cook, Serve, Delicious! 3?!",2020/10/14,2020-10-14,2020
细胞战争,2019/03/30,2019-03-30,2019
Zengeon,2019/06/24,2019-06-24,2019
干支セトラ 陽ノ卷｜干支etc. 陽之卷,2019/01/24,2019-01-24,2019
Jumping Master(跳跳大咖),2019/04/8,2019-04-08,2019
Cube Defender,2019/01/6,2019-01-06,2019
Tower of Origin2-Worm's Nest,2021/09/9,2021-09-09,2021


Les dix premiers jeux affichés ont tous une date complete. Elle a ete correctement convertie mais nous souhaitons toutefois mener la verification sur des dates incompletes egalement.

In [0]:
date_check_df = games_df \
    .groupBy(F.length(F.col("release_date")).alias("date_length")) \
    .agg(F.count("*").alias("games"), F.count("release_date_clean").alias("with_date"), F.count("release_year").alias("with_year")) \
    .orderBy("date_length")

display(date_check_df)


date_length,games,with_date,with_year
0,99,0,0
7,123,0,123
9,15732,15732,15732
10,39737,39737,39737


Le traitement des dates conduit au resultat attendu: les dates complètes ont toutes ete converties, les 123 dates sans jour ont donné une année, les 99 jeux sans date n'ont produit ni date ni année.


### Conversion du nombre de possesseurs

Le nombre de possesseurs (owners) est egalement écrit sous forme de texte. On peut voir dans le tableau qu'il vient sous forme d'une plage de valeur, par exemple: 100,000 .. 200,000.

Afin de traiter cett colonne, nous allons lister toutes la vlaeurs prises par cette colonne.

In [0]:
owners_shapes_df = games_df \
    .groupBy("owners") \
    .count() \
    .orderBy(F.col("count").desc())

display(owners_shapes_df)


owners,count
"0 .. 20,000",38072
"20,000 .. 50,000",7285
"50,000 .. 100,000",3695
"100,000 .. 200,000",2519
"200,000 .. 500,000",2162
"500,000 .. 1,000,000",933
"1,000,000 .. 2,000,000",526
"2,000,000 .. 5,000,000",335
"5,000,000 .. 10,000,000",97
"10,000,000 .. 20,000,000",41


Nous observons 13 valeurs possibles pour cette colonne. La forme generale est: valeur basse, suivie de 2 points, suivit de la valeur haute. Nous pouvons donc scinder ces valeurs en deux.

La conversion de cette colonne va consister à separer les deux partie, retirer la virgule des nombre, et generer deux sorties sous forme de nombre

In [0]:
owners_bounds = F.split(F.regexp_replace(F.col("owners"), ",", ""), " \\.\\. ")

games_df = games_df \
    .withColumn("owners_min", owners_bounds.getItem(0).cast("int")) \
    .withColumn("owners_max", owners_bounds.getItem(1).cast("int"))

display(games_df.select("name", "owners", "owners_min", "owners_max").limit(10))


name,owners,owners_min,owners_max
Counter-Strike,"10,000,000 .. 20,000,000",10000000,20000000
ASCENXION,"0 .. 20,000",0,20000
Crown Trick,"200,000 .. 500,000",200000,500000
"Cook, Serve, Delicious! 3?!","100,000 .. 200,000",100000,200000
细胞战争,"0 .. 20,000",0,20000
Zengeon,"100,000 .. 200,000",100000,200000
干支セトラ 陽ノ卷｜干支etc. 陽之卷,"0 .. 20,000",0,20000
Jumping Master(跳跳大咖),"20,000 .. 50,000",20000,50000
Cube Defender,"0 .. 20,000",0,20000
Tower of Origin2-Worm's Nest,"0 .. 20,000",0,20000


### Nettoyage de la colonne âge requis

L'âge requis (required_age) est egalement sous format texte au lieu d'un nombre. Nous allons la aussi analyser toutes les vlaeurs possibles.


In [0]:
age_shapes_df = games_df \
    .groupBy("required_age") \
    .count() \
    .orderBy(F.col("count").desc())

display(age_shapes_df)


required_age,count
0,55030
15,264
18,223
16,38
17,38
12,32
13,26
14,10
10,7
6,4


Nous observons une certaine variété d'âge avec 21 valeurs differentes. Certaines valeurs sont compréhensibles et pourront être converties en valeurs numériques, par exemple: 7+ ou encore MA 15+. La valeur de 180 ans n'a pas de sens. Nous pouvons suspecter un faute de frappe (18 au lieu de 180) mais elle sera plus simplement ecartée. Nous ecarterons egalement la valeur de 35, probablement la plus difficile à comprendre.
En tout, 5 valeurs sur 55 691 seront ecartées.

In [0]:
extracted_age = F.regexp_extract(F.col("required_age"), "\\d+", 0).cast("int")

games_df = games_df \
    .withColumn("required_age_clean", F.when(extracted_age <= 21, extracted_age))

display(games_df.groupBy("required_age", "required_age_clean").count().orderBy(F.col("count").desc()))


required_age,required_age_clean,count
0,0,55030
15,15,264
18,18,223
16,16,38
17,17,38
12,12,32
13,13,26
14,14,10
10,10,7
180,null,4


### Colonne des type

Nous allons vérifier si cette colonne a besoin d'être nettoyée également. 

In [0]:
display(games_df.groupBy("type").count())


type,count
hardware,1
game,55690


Une seule valeur n'est pas de type game: elle sera donc supprimée. 

In [0]:
games_df = games_df.filter(F.col("type") == "game")
print(games_df.count())


55690


### Cases vides

Les cellules vides ne sont pas des valeurs nulles. Nous allons donc d'abord les compter pour chaque colonne.

In [0]:
missing_df = games_df.select(
    F.count(F.when(F.col("developer") == "", 1)).alias("developer"),
    F.count(F.when(F.col("publisher") == "", 1)).alias("publisher"),
    F.count(F.when(F.col("genre") == "", 1)).alias("genre"),
    F.count(F.when(F.col("languages") == "", 1)).alias("languages"),
    F.count(F.when(F.size(F.col("categories")) == 0, 1)).alias("categories"),
    F.count(F.when(F.col("short_description") == "", 1)).alias("short_description"),
    F.count(F.when(F.col("website") == "", 1)).alias("website")
)

display(missing_df)


developer,publisher,genre,languages,categories,short_description,website
126,132,160,10,970,37,25217


Rapporté à un ensemble de 55690 lignes, le nombre de cases vides est faible. Seuls les websites sont manquants en nombre significatif mais ne sont pas l'objet de notre etude. 

### Table netoyée finalisée

In [0]:
games_df.printSchema()
print(games_df.count())
display(games_df.limit(10))


root
 |-- appid: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- platforms: struct (nullable = true)
 |    |-- linux: boolean (nullable = true)
 |    |-- mac: boolean (nullable = true)
 |    |-- windows: boolean (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- type: string (nullable = true)
 |-- website: string

appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,type,website,price_usd,initial_price_usd,discount_pct,release_date_clean,release_year,owners_min,owners_max,required_age_clean
10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,game,,9.99,9.99,0,2000-11-01,2000,10000000,20000000,0
1000000,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)",0,IndigoBlue Game Studio,0,"Action, Adventure, Indie",https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048,999,"English, Korean, Simplified Chinese",ASCENXION,5,"0 .. 20,000","List(false, false, true)",27,999,PsychoFlux Entertainment,2021/05/14,0,"ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.",game,,9.99,9.99,0,2021-05-14,2021,0,20000,0
1000010,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)",99,NEXT Studios,70,"Adventure, Indie, RPG, Strategy",https://cdn.akamai.steamstatic.com/steam/apps/1000010/header.jpg?t=1655724189,1999,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil",Crown Trick,646,"200,000 .. 500,000","List(false, false, true)",4032,599,"Team17, NEXT Studios",2020/10/16,0,"Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!",game,,5.99,19.99,70,2020-10-16,2020,200000,500000,0
1000030,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)",76,Vertigo Gaming Inc.,0,"Action, Indie, Simulation, Strategy",https://cdn.akamai.steamstatic.com/steam/apps/1000030/header.jpg?t=1660866300,1999,English,"Cook, Serve, Delicious! 3?!",115,"100,000 .. 200,000","List(false, true, true)",1575,1999,Vertigo Gaming Inc.,2020/10/14,0,"Cook, serve and manage your food truck as you dish out hundreds of different foods across war-torn America in this massive sequel to the million-selling series!",game,http://www.cookservedelicious.com,19.99,19.99,0,2020-10-14,2020,100000,200000,0
1000040,List(Single-player),0,DoubleC Games,0,"Action, Casual, Indie, Simulation",https://cdn.akamai.steamstatic.com/steam/apps/1000040/header.jpg?t=1627033870,199,Simplified Chinese,细胞战争,1,"0 .. 20,000","List(false, false, true)",0,199,DoubleC Games,2019/03/30,0,这是一款打击感十足的细胞主题游戏！操作简单但活下去却不简单，“你”作为侵入人体的细菌病毒，通过与细胞之间的战斗来获得基因变异点数和进入下一关的资格，每种细菌病毒都有独特的能力和攻击效果，你是否可以破坏五大器官并占领人体呢！？,game,,1.99,1.99,0,2019-03-30,2019,0,20000,0
1000080,"List(Multi-player, Single-player, Steam Achievements, Full controller support, Steam Trading Cards)",3,IndieLeague Studio,60,"Action, Adventure, Indie, RPG",https://cdn.akamai.steamstatic.com/steam

## 3. Tables par niveau d'analyse

Certaines colonnes contiennent plusieurs informations simultanement, c'est le cas par exemple de 'genre', de 'languages, ou de 'platforms'.

Dasn cette partie nous allons construire des tables qui seront obtenur en 'depliant' la table principale de facon a faire aparaitre chaque valeur de maniere unique. La consequence logique est que les valeurs d'entree seront dupliquee et apparaitront autant de fois qu'il y a d'element d'information dans chaque colonne. par exemple, le jeu ASCENXION a 3 genres, il sera donc deplié en 3 lignes.


### Une ligne par jeu et par genre
Nous allons ici deplier les genres. A l'interieur d'une case ils sont séparés par des virgules. La case est d'abord decoupée en liste, puis on déplie la liste. Les 160 jeux sans genre ne sont pas integrés dans cette table.


In [0]:
game_genres_df = games_df \
    .filter(F.col("genre") != "") \
    .withColumn("genre", F.explode(F.split(F.col("genre"), ", ")))

print(game_genres_df.count())
display(game_genres_df.select("name", "genre").limit(10))


157110


name,genre
Counter-Strike,Action
ASCENXION,Action
ASCENXION,Adventure
ASCENXION,Indie
Crown Trick,Adventure
Crown Trick,Indie
Crown Trick,RPG
Crown Trick,Strategy
"Cook, Serve, Delicious! 3?!",Action
"Cook, Serve, Delicious! 3?!",Indie


Du fait du depliage des genres, nous obtenons desormais un tableau de 157 110 lignes.

### Une ligne par jeu et par langue

On procede de la meme maniere avec les langues que precedement.
La structure est la meme, les langues sont separees par des virgules. Les jeux sans langue precisée ne sont pas integrés.

In [0]:
game_languages_df = games_df \
    .filter(F.col("languages") != "") \
    .withColumn("language", F.explode(F.split(F.col("languages"), ", ")))

print(game_languages_df.count())
display(game_languages_df.groupBy("language").count().orderBy("count"))


202292


language,count
Korean,1
English Dutch English,1
Korean[b]*[/b],1
English (full audio),1
(all with full audio support),1
Maori,1
Spanish - Spain [b]*[/b],1
Macedonian,1
Uzbek,1
Marathi,1



LA table obtenue une fois depliée a 202 292 lignes pour pour 55 680 jeux.
92 valeurs ressortent pour les langues, mais une trentaine environ sont des erreurs de saisie ou problematique synthaxique, par exemple: "Korean[b]*[/b]"


Bien qu'il soit possible de corriger ces erreurs pour l'essntiel des langues, le travail est fastidieux et peu impactant pour l'analyse. Nous ferons le choix de conserver ces données en l'etat.



### Une ligne par jeu et par plateforme

Les plateformes sont un bloc de trois cases vrai ou faux : windows, mac, linux. On fabrique pour chaque jeu la liste des plateformes où il est disponible, puis on la déplie comme pour les genres.


In [0]:
platform_flags = F.array(
    F.when(F.col("platforms.windows"), "windows"),
    F.when(F.col("platforms.mac"), "mac"),
    F.when(F.col("platforms.linux"), "linux")
)

game_platforms_df = games_df \
    .withColumn("platform", F.explode(platform_flags)) \
    .filter(F.col("platform").isNotNull())

print(game_platforms_df.count())
display(game_platforms_df.groupBy("platform").count().orderBy(F.col("count").desc()))


76901


platform,count
windows,55675
mac,12769
linux,8457



La table obtenur compte 76 901 lignes pour 55 690 jeux, soit 1.4 plateformes par jeu.
Ce decompte nous dirige deja vers la reponse à la question des plateformes: presque tous les jeux sont disponibles sous Windows.


## 4. Analyse d'ensemble du marché

Dans ce chapitre nous allons répondre aux questions de niveau macro : les éditeurs, les notes, les parutions par année, les prix et remises, les langues, les limites d'âge. Chaque question est traitée par un calcul, puis un graphique.

### Les éditeurs les plus prolifiques

On compte les jeux par éditeur et on garde les dix premiers. La colonne publisher n'est pas découpée sur les virgules : elles font souvent partie du nom de l'éditeur, comme dans « Big Fish Games, Inc. ». Les 132 jeux sans éditeur sont écartés.


In [0]:
top_publishers_df = games_df \
    .filter(F.col("publisher") != "") \
    .groupBy("publisher") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .limit(10)

display(top_publishers_df)


publisher,count
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
Sekai Project,132
HH-Games,132
Ubisoft,127
Laush Studio,126


Databricks visualization. Run in Databricks to view.

In [0]:
print(games_df.filter(F.col("publisher") != "").select("publisher").distinct().count())


29965


Big Fish Games occupe la première place du classement avec 422 jeux publiés, suivi de 8floor (202) et SEGA (165). Ubisoft est classé 9eme avec 127 jeux.

On constate qu'il y a pres de 30 000 editeurs de jeux pour plus de 55 500 jeux. Le marché est donc tres fragmenté avec un nombre important d'editeur ayant publié un faible nombre de jeux (1 ou 2).


### Les jeux les mieux notés

Le document ne fourni pas de note à proprement parler mais seulement des avis positifs et negatif. Deux colonnes "positive" et "negative" sont données, avec, pour chacune d'elles et pour chaque ligne, le nombre d'avis laissés par les utilisateurs. Il s'agit probablement d'un système de pouce vert / pouce rouge.  L'utilisateur peut soit laisser un pouce vert pour temoigner sa satisfaction, soit mettre un pouce rouge pour temoigner son mecontentement.

Ce type de données est délicat à traiter. La mesure la plus naturelle est de faire le ratio entre le nombre d'avis positif et le nombre d'avis total, mais cette mesure n'a aucune signification pour un faible nombre d'avis.
Nous allons donc evaluer le nombre d'avis total (avis positifs + avis negatifs) pour differents seuils:
- aucun
- au moins 10
- au moins 100
- au moins 500
- au moins 1000

Nous allons aussi nous interesser à la mediane du nombre d'avis.


In [0]:
games_df = games_df.withColumn("reviews", F.col("positive") + F.col("negative"))

display(games_df.select(
    F.count(F.when(F.col("reviews") == 0, 1)).alias("no_review"),
    F.count(F.when(F.col("reviews") >= 10, 1)).alias("at_least_10"),
    F.count(F.when(F.col("reviews") >= 100, 1)).alias("at_least_100"),
    F.count(F.when(F.col("reviews") >= 500, 1)).alias("at_least_500"),
    F.count(F.when(F.col("reviews") >= 1000, 1)).alias("at_least_1000"),
    F.median("reviews").alias("median_reviews")
))


no_review,at_least_10,at_least_100,at_least_500,at_least_1000,median_reviews
163,38999,16332,7767,5311,26.0


La moitié des jeux ont moins de 26 avis et 163 n'en ont aucun. certains jeu, ayant peu d'avis ont 100% d'avis positifs. Aucun avis négatif n'est encore arrivé. Pour fixer un seuil, nous allons chercher le jeu le plus commenté qui n'a encore aucun avis négatif.



In [0]:
max_reviews_without_negative = games_df.filter(F.col("negative") == 0).agg(F.max("reviews")).take(1)[0][0]
print(max_reviews_without_negative)


496


Le jeu ayant reçu le plus d'avis, tous positifs, a reçu 496 avis. Au-delà de ce nombre, tous les jeux ont au moins un avis négatif. 
Nous allons donc utiliser ce nombre en tant que seuil au-delà duquel nous classerons les jeux. 


In [0]:
games_df = games_df.withColumn("positive_share", F.when(F.col("reviews") > 0, F.col("positive") / F.col("reviews")))

top_rated_df = games_df \
    .filter(F.col("reviews") > max_reviews_without_negative) \
    .select("name", "positive", "negative", "reviews", F.round(F.col("positive_share") * 100, 2).alias("positive_pct")) \
    .orderBy(F.col("positive_share").desc()) \
    .limit(10)

display(top_rated_df)


name,positive,negative,reviews,positive_pct
Flowers -Le volume sur ete-,937,1,938,99.89
Touhou Fuujinroku ~ Mountain of Faith.,649,4,653,99.39
Aventura Copilului Albastru și Urât,2203,14,2217,99.37
The Case of the Golden Idol,909,6,915,99.34
Aseprite,11823,80,11903,99.33
Patrick's Parabox,1489,11,1500,99.27
A Short Hike,11645,87,11732,99.26
Monster Prom 3: Monster Roadtrip,1052,8,1060,99.25
Flowers -Le volume sur automne-,516,4,520,99.23
CULTIC,2021,16,2037,99.21


### Les parutions par année

Nous allons compter le nombre de jeux par année de parution. Nous allons egalement verifier si la derniere année apparaissant dans les données est comptabilisée de maniere entiere ou non.


In [0]:
print(games_df.agg(F.max("release_date_clean")).take(1)[0][0])

releases_by_year_df = games_df \
    .filter(F.col("release_year").isNotNull()) \
    .groupBy("release_year") \
    .count() \
    .orderBy("release_year")

display(releases_by_year_df)


2022-11-11


release_year,count
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


Databricks visualization. Run in Databricks to view.

Les données s'arrêtent au 11 novembre 2022. la dernière année est donc incomplète. Nous observons que moins de 500 jeux par an sont publiés jusqu'en 2013. S'en suit une forte croissance à partir de 2014 allant jusqu'à 7 678 jeux en 2018. Après un léger ralentissement de la croissance en 2019 (6 968), les deux années de Covid constituent les records du catalogue : 8 305 jeux en 2020 et 8 823 en 2021. La periode du covid coincide avec les volumes les plus elevés de parution.


### Les prix et les remises

Nous allons nous interesser à la distribution des prix. Nous allons notament calculer le nombre de jeux gratuits, le prix médian, le prix moyen et le prix maximal. On compte aussi les jeux en promotion et la remise médiane parmi eux.


In [0]:
display(games_df.select(
    F.count(F.when(F.col("price_usd") == 0, 1)).alias("free_games"),
    F.median("price_usd").alias("median_price"),
    F.round(F.avg("price_usd"), 2).alias("mean_price"),
    F.max("price_usd").alias("max_price"),
    F.count(F.when(F.col("discount_pct") > 0, 1)).alias("discounted_games"),
    F.median(F.when(F.col("discount_pct") > 0, F.col("discount_pct"))).alias("median_discount")
))


free_games,median_price,mean_price,max_price,discounted_games,median_discount
7779,4.99,7.73,999.0,2518,60.0


7 779 jeux sont gratuits, soit 14 % du catalogue. 
La moitié des jeux coûtent moins de 4,99 dollars, et la moyenne des prix est de 7,73 dollars. Cet ecart entre moyenne et mediane s'explique par quelques jeux coutant très chers, le plus cher etant à 999 dollars. Les promotions ne concernent que 2 518 jeux, soit 4,5 %, la remise médiane est de 60 %.


Nous allons observer la forme de la distribution. Pour cela nous devons tout d'abord ranger les jeux par tranche de prix.


In [0]:
price_band = F.when(F.col("price_usd") == 0, "0. gratuit") \
    .when(F.col("price_usd") < 5, "1. moins de 5 $") \
    .when(F.col("price_usd") < 10, "2. de 5 à 10 $") \
    .when(F.col("price_usd") < 20, "3. de 10 à 20 $") \
    .when(F.col("price_usd") < 40, "4. de 20 à 40 $") \
    .when(F.col("price_usd") < 60, "5. de 40 à 60 $") \
    .otherwise("6. 60 $ et plus")

price_bands_df = games_df \
    .groupBy(price_band.alias("price_band")) \
    .count() \
    .orderBy("price_band")

display(price_bands_df)


price_band,count
0. gratuit,7779
1. moins de 5 $,23478
2. de 5 à 10 $,12450
3. de 10 à 20 $,9022
4. de 20 à 40 $,2394
5. de 40 à 60 $,445
6. 60 $ et plus,122


Databricks visualization. Run in Databricks to view.

Nous constatons que plus de la moitié (56%) du catalogue est à moins de 5 dollars, voire meme gratuit pour 14% des jeux. Seulement 567 jeux dépassent 40 dollars, soit environt 1 %.


### Les langues les plus représentées

On utilise la table à une ligne par jeu et par langue : on compte les jeux pour chaque langue et on garde les dix premières.


In [0]:
top_languages_df = game_languages_df \
    .groupBy("language") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .limit(10)

display(top_languages_df)


language,count
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6599


Databricks visualization. Run in Databricks to view.

L'anglais est présent dans 55 116 jeux, soit 99 % des 55 680 jeux qui déclarent une langue : c'est la langue de référence de la plateforme. Loin derrière, un groupe de cinq langues se tient entre 12 000 et 14 000 jeux, soit environ un quart du catalogue : l'allemand, le français, le russe, le chinois simplifié et l'espagnol. Le français figure dans 13 426 jeux, soit 24 %.


### Les limites d'âge

On compte les jeux par âge requis, avec la colonne nettoyée, puis les jeux interdits aux moins de 16 ans et aux moins de 18 ans.


In [0]:
display(games_df.select(
    F.count(F.when(F.col("required_age_clean") == 0, 1)).alias("no_age_limit"),
    F.count(F.when(F.col("required_age_clean") > 0, 1)).alias("with_age_limit"),
    F.count(F.when(F.col("required_age_clean") >= 16, 1)).alias("at_least_16"),
    F.count(F.when(F.col("required_age_clean") >= 18, 1)).alias("at_least_18"),
    F.count(F.when(F.col("required_age_clean").isNull(), 1)).alias("unknown")
))


no_age_limit,with_age_limit,at_least_16,at_least_18,unknown
55029,656,301,225,5


Seulement 656 jeux sur 55 690 ont une limite d'âge déclarée, soit 1,2 %. Parmi eux, 301 sont interdits aux moins de 16 ans et 225 aux moins de 18 ans, soit 0,4 % du catalogue. La quasi-totalité des jeux, 55 029, n'affiche aucune restriction. 

## 5. Analyse des genres

Dans ce chapitre nous allons utiliser la table à une ligne par jeu et par genre. Un jeu à plusieurs genres sera present dans autant de ligne qu'il posede de genre.

### Les genres les plus représentés

Nous allons compter combien de fois chaque genre existe dans la table entiere.


In [0]:
genre_counts_df = game_genres_df \
    .groupBy("genre") \
    .count() \
    .orderBy(F.col("count").desc())

display(genre_counts_df)


genre,count
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


Databricks visualization. Run in Databricks to view.

Le genre Indie est le plus fréquent avec 39 681 jeux, soit 71 % des jeux qui ont un genre. Il est suivi de:
- Action (23 759)
- Casual (22 086)
- Adventure (21 431)


### La part d'avis positifs par genre

Sur le meme modele que defini precedement pour le comptage des avis, pour chaque genre, d'une part on additionne les avis positifs, d'autre part on additionne tous les avis et on fait le raport entre les deux.

In [0]:
genre_reviews_df = game_genres_df \
    .groupBy("genre") \
    .agg(F.count("*").alias("games"), F.sum("positive").alias("positive"), F.sum(F.col("positive") + F.col("negative")).alias("reviews")) \
    .withColumn("positive_pct", F.round(F.col("positive") / F.col("reviews") * 100, 1)) \
    .orderBy(F.col("positive_pct").desc())

display(genre_reviews_df)



genre,games,positive,reviews,positive_pct
Photo Editing,105,577751,591496,97.7
Animation & Modeling,322,690765,717157,96.3
Design & Illustration,406,674057,701064,96.1
Utilities,682,739335,782838,94.4
Game Development,159,27461,30735,89.3
Indie,39681,32531023,36772257,88.5
Audio Production,195,69118,78546,88.0
Video Production,247,111514,127876,87.2
Casual,22086,10034967,11572263,86.7
Web Publishing,89,33984,39222,86.6


Databricks visualization. Run in Databricks to view.

Les 4 premiers elements du classement sont:
- Photo Editing
- Animation & Modeling
- Design & Illustration 
- Utilities

Ce ne sont pas des genres de jeu mais des types de logiciels. Ils sont présent en tant que game mais ce n'est pas le cas.


En triant le tableau par nombre de jeux, on constate que ces catégories, comme toutes celles qui ne sont pas des genres de jeu (Education, Video Production, Violent, Accounting…), comptent moins de 700 titres, alors que le plus petit genre de jeu, Massively Multiplayer, en compte 1 460. Aucune catégorie ne se situe entre les deux. Un filtre à 1 000 jeux, au milieu de ce trou, sépare donc exactement les genres de jeu des autres catégories.



In [0]:
main_genre_reviews_df = genre_reviews_df.filter(F.col("games") >= 1000)

display(main_genre_reviews_df)


genre,games,positive,reviews,positive_pct
Indie,39681,32531023,36772257,88.5
Casual,22086,10034967,11572263,86.7
Simulation,10836,15572390,17972902,86.6
Racing,2155,2340353,2724044,85.9
RPG,9534,19425528,22699856,85.6
Action,23759,54858618,64546277,85.0
Strategy,10895,13402870,15796295,84.8
Adventure,21431,29689445,35342598,84.0
Early Access,6145,4334595,5270786,82.2
Free to Play,3393,18722246,23003053,81.4


Databricks visualization. Run in Databricks to view.

Apres filtrage il reste 12 genres de jeux et nous constatons que les écarts entre ces différents genres sont relativement faibles, le maximum étant de 88,5 % et le minimum de 73.1 %. 



## 6. Analyse des plateformes

Le comptage par plateforme a été fait au chapitre 3 : sur 55 690 jeux, 55 675 sont disponibles sous Windows (99,97 %), 12 769 sous Mac (23 %) et 8 457 sous Linux (15 %). Windows est la plateforme par défaut ; la question est de savoir si certains genres sont plus souvent portés sur Mac et Linux que d'autres.

### La disponibilité sur Mac et Linux par genre

Pour chacun des douze genres de jeu, on calcule la part de ses jeux disponibles sur Mac, puis sur Linux.


In [0]:
genre_platforms_df = game_genres_df \
    .groupBy("genre") \
    .agg(
        F.count("*").alias("games"),
        F.round(F.avg(F.col("platforms.mac").cast("int")) * 100, 1).alias("mac_pct"),
        F.round(F.avg(F.col("platforms.linux").cast("int")) * 100, 1).alias("linux_pct")
    ) \
    .filter(F.col("games") >= 1000) \
    .orderBy(F.col("mac_pct").desc())

display(genre_platforms_df)


genre,games,mac_pct,linux_pct
Strategy,10895,27.6,16.8
Indie,39681,25.0,17.6
Free to Play,3393,24.9,14.0
RPG,9534,23.6,16.0
Adventure,21431,23.5,15.4
Casual,22086,23.2,15.0
Simulation,10836,22.5,14.1
Racing,2155,19.7,14.1
Action,23759,19.2,14.2
Sports,2666,19.0,10.8


Databricks visualization. Run in Databricks to view.

Nous pouvons observer qu'aucun genre n'est majoritaire sur Mac ou Linux. la part va de 14,6 % à 27,6 % pour Mac et de 10,3 % à 17,6 % pour Linux. 
Les jeux les plus representés sur mac sont des jeux de stratergie ou Indie.
Pour Ubisoft, Windows est la plateforme privilegiée, le developpement du jeu pour sur des support Mac et Linux est plutot adapté à des jeux de stratégie et indépendants.


## 7. Synthèse pour Ubisoft

A la date d'arret du decompte de ce catalogue, Steam compte 55 690 jeux. La majeure partie d'entre eux (94%) sont parus apres 2014, date qui peut etre qualifiée de 'percée' pour la plateforme. Les années 2020 et 2021 correspondant aux année de Covid, ont connues les plus fortes croissance en nombre de parutions, depassant 8 000  nouveaux jeux chacune. 

Le marché est essentiellement constitué de petis editeurs independant. On retrouve cette information à deux reprise dans l'analyse: D'une part prés de 30 000 editeurs pour plus de 55 500 jeux impose mathematiquement que la pluspart d'entre eux n'en ont diffusé qu'un seul ; d'autre part le genre indie (independant), present pour pres de 40 000 jeux viens confirmer ce point. Les dix premiers acteurs ne representent d'ailleur que 3 % du catalogue. Ubisoft occupe la neuvième place du classement en nombre de jeux avec 127 jeux. Plus de la moitié des jeux coûtent moins de 5$, 14 % sont meme gratuits. Seulemenr 1 % dépassent 40$, le plus cher etant à 999$. Nous pouvons donc raisonablement qualifier ces prix de 'bas' en general avec un marché de niche pour les productions plus consequentes.

Concernant la popularité,les jeux ayant reçus le plus d'avis positifs (en pourcentage) relativement à la totalité des avis sont les jeux de type 'indie'. Ils ont recu 88.5% d'avis positifs. Les dix jeux les mieux notés, tous genres confondus, dépassent même 99 %. En consequent les jeux les mieux notés ne viennent pas necessairement des grosses productions et la popularité ne se confond pas toujours avec la taille du publisher et leur possible investissement en production.

Le genre n'a que peu d'incidence sur l'avis des joureurs. Les jeux sont globalement bien reçus, entre 80 et 89 % d'avis positifs, avec une exception : les jeux massivement multijoueurs, à 73 %, sont les moins bien accueillis.

Coté langue, l'anglais domine largement. Elle est presente dans 99 % des jeux.  Les principales langues suivantes sont dans l'ordre:
- l' allemand
- le francais
- le russe
- le chinois
- l'espagnol
- le japomais
- l'italien


Au niveau des OS, Windows domine largement et est disponible pour 99,97 % des jeux, Mac suit avec 23 % et Linux pour 15 %, les portages étant surtout le fait des jeux de stratégie et indépendants.

De cela nous pouvons conclure qu'un nouveau jeu, pour etre idealiement positioné, doit prevoir:
- un OS windows à minima
- Langue anglaise a minima, preferablement les suivantes ici listées
- un prix faible (typiquement 5$)

Le genre 'massivement multijoueur' est preferablement à eviter.
Le positionnement dans une categorie limitant l'age est egalement niche.


Limites de l'analyse :

Cette etude ne porte pas sur les ventes elles meme mais sont seulement une observation du pannel d'offre de steam au moment du relevé des données.
